In [3]:
import pandas as pd
from pathlib import Path


BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
OUTPUT_DIR = BASE_DIR.parent / "data" / "raw"
input_file = "/home/kene//Downloads/online_retail_II/online_retail_II.xlsx"

# Read all sheets (UCI Online Retail II contains 'Year 2009-2010' and 'Year 2010-2011')
xls = pd.ExcelFile(input_file)
df = pd.concat([pd.read_excel(xls, sheet) for sheet in xls.sheet_names], ignore_index=True)


# Standardize column naming
df.columns = df.columns.str.strip().str.replace(" ", "")
df["Invoice"] = df["Invoice"].astype(str).str.strip()
df["StockCode"] = df["StockCode"].astype(str).str.strip()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


# Rule: Filter valid non-empty descriptions, clean text, and pick
# the most frequently occurring (mode) description per StockCode.
prod_clean = df.dropna(subset=["Description"]).copy()
prod_clean["Description"] = prod_clean["Description"].str.strip().str.upper()

products_df = (
    prod_clean.groupby(["StockCode", "Description"], as_index=False)
    .size()
    .sort_values(["StockCode", "size"], ascending=[True, False])
    .drop_duplicates(subset=["StockCode"], keep="first")[["StockCode", "Description"]]
)
products_df.to_csv(OUTPUT_DIR / "products.csv", index=False)
print(f"Exported products.csv ({len(products_df):,} rows)")


# Rule: Isolate valid Customer IDs, resolve any multi-country entries
# by taking the most frequent country, and export to JSON format.
cust_clean = df.dropna(subset=["CustomerID"]).copy()
cust_clean["CustomerID"] = cust_clean["CustomerID"].astype(int)

customers_df = (
    cust_clean.groupby(["CustomerID", "Country"], as_index=False)
    .size()
    .sort_values(["CustomerID", "size"], ascending=[True, False])
    .drop_duplicates(subset=["CustomerID"], keep="first")[["CustomerID", "Country"]]
)
customers_df.to_json(OUTPUT_DIR / "customers.json", orient="records", indent=2)
print(f"Exported customers.json ({len(customers_df):,} records)")


# Separate valid purchases (positive quantities, non-'C' invoices)
is_return = df["Invoice"].str.startswith("C", na=False) | (df["Quantity"] <= 0)
sales_df = df[~is_return].copy()

# Generate surrogate primary key (sale_id) for relational integrity
sales_df.sort_values("InvoiceDate", inplace=True)
sales_df.reset_index(drop=True, inplace=True)
sales_df["sale_id"] = sales_df.index + 1

# Column selection for the fact table
sales_fact = sales_df[[
    "sale_id", "Invoice", "StockCode", "Quantity", "Price", "InvoiceDate", "CustomerID"
]]
sales_fact.to_csv(OUTPUT_DIR / "sales.csv", index=False)
print(f"Exported sales.csv ({len(sales_fact):,} rows)")


# Extract return rows and convert negative quantities to absolute values
returns_df = df[is_return].copy()
returns_df["Quantity"] = returns_df["Quantity"].abs()
returns_df.sort_values("InvoiceDate", inplace=True)
returns_df.reset_index(drop=True, inplace=True)
returns_df["return_id"] = returns_df.index + 1

# Match cancellations to nearest prior purchase (same CustomerID + StockCode)
# Drop null customer rows first as they cannot be matched back to a purchase
returns_matchable = returns_df.dropna(subset=["CustomerID"]).copy()
returns_matchable["CustomerID"] = returns_matchable["CustomerID"].astype(int)

sales_matchable = sales_df.dropna(subset=["CustomerID"]).copy()
sales_matchable["CustomerID"] = sales_matchable["CustomerID"].astype(int)

# Use pd.merge_asof with exact match on CustomerID + StockCode
matched_returns = pd.merge_asof(
    returns_matchable,
    sales_matchable[["sale_id", "CustomerID", "StockCode", "InvoiceDate"]],
    on="InvoiceDate",
    by=["CustomerID", "StockCode"],
    direction="backward",  # Finds the nearest prior sale date
    suffixes=("", "_sale") 
)

returns_final = matched_returns[[
    "return_id", "sale_id", "Invoice", "StockCode", "Quantity", "Price", "InvoiceDate", "CustomerID"
]]
returns_final.to_csv(OUTPUT_DIR / "returns.csv", index=False)
print(f"Exported returns.csv ({len(returns_final):,} rows)")

Exported products.csv (4,949 rows)
Exported customers.json (5,942 records)
Exported sales.csv (1,044,420 rows)
Exported returns.csv (18,744 rows)
